# Data Preprocessing Pipeline (Beginner)

Raw data is almost never ready for a model. It has **holes** (missing values), **words where numbers are expected** (categorical text), and **features on wildly different scales**. This notebook walks through the three core cleaning steps, one at a time, and then bolts them together into a single, leakage-safe object.

1. **Missing values** — fill the gaps (imputation) instead of throwing away rows.
2. **Encoding** — turn categories into numbers. *Label* encoding for **ordered** categories, *one-hot* encoding for **unordered** ones.
3. **Scaling** — put numeric features on a comparable scale (`StandardScaler` vs `MinMaxScaler`).

The golden rule that ties it all together: **learn the cleaning recipe from the training data only, then apply it to test data.** Peeking at the test set while cleaning is called *data leakage* and it quietly inflates your scores. `scikit-learn`'s `Pipeline` + `ColumnTransformer` enforce this for you.

In [ ]:
import numpy as np                # numeric arrays and the random seed
import pandas as pd               # the DataFrame: our table of messy data
import matplotlib.pyplot as plt   # plotting
import seaborn as sns             # prettier statistical plots (histograms here)

# scikit-learn preprocessing building blocks:
from sklearn.impute import SimpleImputer               # fills in missing values
from sklearn.preprocessing import (
    OneHotEncoder,      # nominal (unordered) categories -> indicator columns
    OrdinalEncoder,     # ordinal (ordered) categories   -> ordered integers
    StandardScaler,     # scale to mean 0 / std 1
    MinMaxScaler,       # scale into the [0, 1] range
)
from sklearn.compose import ColumnTransformer          # apply different steps to different columns
from sklearn.pipeline import Pipeline                  # chain steps into one object
from sklearn.model_selection import train_test_split   # split into train/test

# One seed to rule them all: makes the synthetic data (and everything downstream) reproducible.
SEED = 42
rng = np.random.default_rng(SEED)   # modern NumPy random generator, used to build the toy dataset

sns.set_theme(style="whitegrid")    # consistent plot styling
pd.set_option("display.width", 100)
print("setup complete")

## 1. Build a small, messy dataset

Rather than download anything, we **synthesize** a table so the notebook runs fully offline. Imagine rows describing customers:

| column | type | notes |
|---|---|---|
| `age` | numeric | some values missing (NaN) |
| `income` | numeric | some values missing (NaN) |
| `size` | **ordinal** categorical | `S < M < L < XL` — the order is *meaningful* |
| `city` | **nominal** categorical | `Pune / Mumbai / Delhi` — no natural order |
| `purchased` | target | 0/1 label we would eventually predict |

The key distinction to keep in mind: **`size` is ordinal** (S is genuinely *less than* L), while **`city` is nominal** (Mumbai is not "more than" Pune). That single fact decides how we encode each one later.

In [ ]:
n = 200   # number of rows

# --- numeric columns, drawn from simple distributions ---
age = rng.normal(loc=40, scale=12, size=n).round(0)          # ages centered around 40
income = rng.normal(loc=60000, scale=15000, size=n).round(0) # incomes centered around 60k

# --- categorical columns, drawn from fixed category lists ---
size = rng.choice(["S", "M", "L", "XL"], size=n, p=[0.3, 0.4, 0.2, 0.1])   # ORDINAL: has a natural order
city = rng.choice(["Pune", "Mumbai", "Delhi"], size=n, p=[0.4, 0.35, 0.25]) # NOMINAL: no order

# --- target: a 0/1 label loosely related to the features (just so it's not random noise) ---
score = (income / 60000) + (age < 35).astype(float) * 0.5 + rng.normal(0, 0.3, size=n)
purchased = (score > np.median(score)).astype(int)

df = pd.DataFrame({
    "age": age,
    "income": income,
    "size": size,
    "city": city,
    "purchased": purchased,
})

# --- now punch some holes: set a handful of numeric AND categorical values to NaN ---
# We pick random row indices for each column and blank them out, so the data looks realistically messy.
df.loc[rng.choice(n, size=18, replace=False), "age"] = np.nan       # ~9% of ages missing
df.loc[rng.choice(n, size=12, replace=False), "income"] = np.nan    # ~6% of incomes missing
df.loc[rng.choice(n, size=10, replace=False), "size"] = np.nan      # a few sizes missing (categorical NaN)

df.head(10)

### Look at the mess before touching it

`df.info()` shows the dtype and the **non-null count** per column — anything below the total row count has missing values. `df.isna().sum()` counts the NaNs directly. *Always* inspect this first: it tells you which columns need imputation and whether a column is numeric or text.

In [ ]:
print("=== df.info() ===")
df.info()                       # dtypes + non-null counts (the gaps show up as reduced counts)

print("\n=== missing values per column ===")
print(df.isna().sum())          # explicit NaN count per column

print("\n=== numeric summary (note count < 200 where values are missing) ===")
print(df.describe().round(1))   # count/mean/std/min/max for numeric columns

## 2. Split FIRST, clean second

Before we impute or scale anything, we split into **train** and **test**. Why now? Because every cleaning statistic — the median we fill with, the mean/std we scale by, the list of categories we one-hot — must be learned from **training data only**. If we computed a median over the whole dataset (including test rows) we'd be leaking future information into the model, and our test score would be optimistically wrong.

So the discipline is: **split → `fit` on train → `transform` both.**

In [ ]:
X = df.drop(columns="purchased")   # features (still messy: NaNs + text categories)
y = df["purchased"]                # target

# stratify=y keeps the 0/1 ratio the same in both splits; random_state makes the split reproducible.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=SEED, stratify=y
)

print(f"train rows: {len(X_train)},  test rows: {len(X_test)}")
print(f"missing in train:\n{X_train.isna().sum()}")

## 3. Missing values — impute, don't drop

The lazy fix is `df.dropna()` (delete any row with a NaN). It's usually **wasteful**: here a single missing `age` would throw away that customer's perfectly good `income`, `city`, and `size` too. Drop enough rows and you can lose a large fraction of your data — and if the missingness isn't random you also *bias* what's left.

The better default is **imputation**: fill each gap with a sensible substitute.

- **Numeric columns** → fill with the **median** (robust to outliers, unlike the mean).
- **Categorical columns** → fill with the **most frequent** category (the mode).

Crucially, the median/mode are computed on **train only** (`fit`), then applied to test (`transform`). `SimpleImputer` does exactly this. Below we demonstrate it on the numeric columns in isolation so you can see the gaps disappear.

In [ ]:
# How many rows would dropping cost us? (motivation for imputing instead)
n_dropped = X_train.shape[0] - X_train.dropna().shape[0]
print(f"dropna() would delete {n_dropped} of {X_train.shape[0]} training rows "
      f"({n_dropped / X_train.shape[0]:.0%}) — wasteful!\n")

num_cols = ["age", "income"]   # the numeric columns

# median imputer: learns one median PER COLUMN from the training data
num_imputer = SimpleImputer(strategy="median")
num_imputer.fit(X_train[num_cols])                      # fit = learn medians on TRAIN ONLY
print("learned medians (from train):",
      dict(zip(num_cols, num_imputer.statistics_.round(1))))

# transform both splits using those train medians
train_num_filled = num_imputer.transform(X_train[num_cols])
test_num_filled = num_imputer.transform(X_test[num_cols])   # TEST reuses TRAIN's medians -> no leakage
print("NaNs remaining in train numeric after impute:",
      int(np.isnan(train_num_filled).sum()))

# categorical imputer: fills missing 'size' with the most common size in train
cat_imputer = SimpleImputer(strategy="most_frequent")
cat_imputer.fit(X_train[["size"]])
print("most-frequent 'size' (from train):", cat_imputer.statistics_[0])

## 4. Encoding categories — label vs one-hot

Models do math on numbers, not on the strings `"M"` or `"Mumbai"`. We must convert categories to numbers — but **how** we convert depends on whether the category has a natural order.

### Label / ordinal encoding — for ORDERED categories
Map each category to an integer that **respects the order**: `S=0, M=1, L=2, XL=3`. This is perfect for `size` because the numeric gaps encode a real fact (XL > L > M > S). We pass the categories in order explicitly so the mapping isn't left to chance.

### One-hot encoding — for UNORDERED categories
For `city` there is **no** order, so we must **not** write `Pune=0, Mumbai=1, Delhi=2`. Doing that would tell the model a lie: that `Delhi (2)` is "twice" `Mumbai (1)`, and that Mumbai sits "between" Pune and Delhi. That injected **fake ordering** distorts distances and linear coefficients.

Instead, one-hot encoding creates one 0/1 **indicator column per category** (`city_Pune`, `city_Mumbai`, `city_Delhi`). Exactly one is 1 per row; no ordering is implied. The cost is extra columns, but the categories stay honest.

In [ ]:
# --- LABEL / ORDINAL encoding for the ORDERED 'size' column ---
# We first impute 'size' (reuse the most-frequent imputer above) so there are no NaNs to encode.
size_train = cat_imputer.transform(X_train[["size"]])   # fill missing sizes with train mode

# categories=[...] fixes the ORDER, so the integers are meaningful: S<M<L<XL -> 0<1<2<3
ordinal_enc = OrdinalEncoder(categories=[["S", "M", "L", "XL"]])
size_encoded = ordinal_enc.fit_transform(size_train)

print("ordinal mapping (order is meaningful):", {c: i for i, c in enumerate(["S", "M", "L", "XL"])})
print("first 8 sizes  :", size_train[:8, 0].tolist())
print("encoded as     :", size_encoded[:8, 0].astype(int).tolist())

print("\n" + "-" * 60 + "\n")

# --- ONE-HOT encoding for the UNORDERED 'city' column ---
# sparse_output=False returns a plain dense array; handle_unknown='ignore' is safe if test has
# a city unseen in train (it just produces all-zeros for that row rather than crashing).
onehot_enc = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
city_encoded = onehot_enc.fit_transform(X_train[["city"]])   # fit categories from TRAIN

# get_feature_names_out shows the new indicator column names
city_cols = onehot_enc.get_feature_names_out(["city"])
print("one-hot columns:", list(city_cols))
print("first 5 cities :", X_train["city"].head(5).tolist())
print(pd.DataFrame(city_encoded[:5], columns=city_cols).astype(int).to_string(index=False))

Notice the contrast: `size` collapses to **one** ordered integer column (the order carries information), while `city` expands to **three** 0/1 columns (no order, so each category gets its own flag). Using label encoding on `city` would have been a modelling bug even though the code would run fine — the lesson is that *correct preprocessing depends on the meaning of the data, not just its type.*

## 5. Scaling numeric features — StandardScaler vs MinMaxScaler

`age` lives around 20–60 while `income` lives around 60,000. Many algorithms (linear/logistic regression, SVMs, k-NN, k-means, neural nets) are sensitive to scale: the large-magnitude `income` would dominate distance and gradient calculations purely because of its units. **Scaling** puts every feature on a comparable footing.

**StandardScaler** — subtract the mean, divide by the std, giving each feature **mean 0, std 1**:
$$x' = \frac{x - \mu}{\sigma}$$
Values are centered but unbounded. This is the sensible **default** for most models and copes well with roughly-Gaussian features.

**MinMaxScaler** — squash into a fixed range, usually **[0, 1]**:
$$x' = \frac{x - x_{\min}}{x_{\max} - x_{\min}}$$
Great when you need **bounded** inputs (e.g. image pixels, or neural-net inputs where a fixed range helps), but it is **sensitive to outliers** — a single extreme value stretches everyone else toward 0.

As always, `μ, σ` (or `min, max`) are learned on **train only**.

In [ ]:
# Work on the imputed numeric columns from step 3 (no NaNs left to scale).
std_scaler = StandardScaler()
minmax_scaler = MinMaxScaler()

train_std = std_scaler.fit_transform(train_num_filled)     # learn mean/std on train, transform train
train_mm = minmax_scaler.fit_transform(train_num_filled)   # learn min/max on train, transform train

# Compare summary stats BEFORE and AFTER each scaler (income column shown).
summary = pd.DataFrame({
    "raw_income":       pd.Series(train_num_filled[:, 1]).describe(),
    "standard_income":  pd.Series(train_std[:, 1]).describe(),
    "minmax_income":    pd.Series(train_mm[:, 1]).describe(),
}).round(3)
print(summary)
print("\nStandardScaler -> mean ~0, std ~1.   MinMaxScaler -> min 0, max 1.")

In [ ]:
# Plot the 'income' distribution under each scaling so the shape-vs-range effect is visible.
fig, ax = plt.subplots(1, 3, figsize=(13, 3.5))

sns.histplot(train_num_filled[:, 1], bins=20, ax=ax[0], color="#888888")
ax[0].set_title("Raw income\n(range ~ tens of thousands)"); ax[0].set_xlabel("income")

sns.histplot(train_std[:, 1], bins=20, ax=ax[1], color="#6699cc")
ax[1].set_title("StandardScaler\n(centered at 0, unbounded)"); ax[1].set_xlabel("z-score")

sns.histplot(train_mm[:, 1], bins=20, ax=ax[2], color="#cc6666")
ax[2].set_title("MinMaxScaler\n(squashed into [0, 1])"); ax[2].set_xlabel("scaled value")

# The SHAPE of the distribution is identical in all three — scaling only changes the
# x-axis (location/spread), never the relative arrangement of the points.
plt.tight_layout(); plt.show()

## 6. Tie it together — one leakage-safe Pipeline

Doing impute → encode → scale by hand for every column is error-prone: it's easy to forget to reuse a train statistic, or to apply a step to the wrong column. `scikit-learn` solves this with two objects:

- **`Pipeline`** chains steps for a group of columns (e.g. *impute then scale*).
- **`ColumnTransformer`** routes different column groups to different pipelines and glues the results back together.

We build three lanes:

| columns | steps |
|---|---|
| `age`, `income` (numeric) | median-impute → StandardScaler |
| `size` (ordinal) | most-frequent-impute → OrdinalEncoder(S<M<L<XL) |
| `city` (nominal) | most-frequent-impute → OneHotEncoder |

The payoff: the whole recipe becomes **one object**. Calling `fit_transform` on the training data learns *every* statistic (medians, modes, means, category lists) at once, and a later `transform(X_test)` replays them — **leakage-safe by construction**. The same object drops straight into a model pipeline or cross-validation with no re-plumbing.

In [ ]:
num_cols = ["age", "income"]   # numeric lane
ord_cols = ["size"]            # ordinal lane
nom_cols = ["city"]            # nominal lane

# Lane 1: numeric -> fill NaN with median, then standard-scale
numeric_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])

# Lane 2: ordinal -> fill NaN with most frequent, then encode with the meaningful order
ordinal_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("encode", OrdinalEncoder(categories=[["S", "M", "L", "XL"]])),
])

# Lane 3: nominal -> fill NaN with most frequent, then one-hot
nominal_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("encode", OneHotEncoder(sparse_output=False, handle_unknown="ignore")),
])

# ColumnTransformer: send each column group down its own lane
preprocessor = ColumnTransformer([
    ("num", numeric_pipe, num_cols),
    ("ord", ordinal_pipe, ord_cols),
    ("nom", nominal_pipe, nom_cols),
])

# fit_transform on TRAIN learns every statistic; transform on TEST replays them (no leakage).
X_train_ready = preprocessor.fit_transform(X_train)
X_test_ready = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()   # readable names for the output columns
print("output shape (train):", X_train_ready.shape)
print("output columns:", list(feature_names))

In [ ]:
# Wrap the transformed array in a DataFrame just to view it nicely.
ready_df = pd.DataFrame(X_train_ready, columns=feature_names)

print("Fully preprocessed training data (first 5 rows):")
print(ready_df.head().round(3).to_string(index=False))

# Sanity checks that prove the recipe worked end-to-end:
print("\nNaNs remaining anywhere:", int(np.isnan(X_train_ready).sum()))          # -> 0, all gaps filled
print("numeric 'age' mean ~0? ", round(ready_df['num__age'].mean(), 3),
      " std ~1? ", round(ready_df['num__age'].std(), 3))                         # StandardScaler worked
print("one-hot city columns present:",
      [c for c in feature_names if c.startswith('nom__city')])                    # nominal expanded
print("ordinal size is a single ordered column:",
      [c for c in feature_names if c.startswith('ord__')])                        # ordinal stayed 1 col

## Recap

| step | what | why |
|---|---|---|
| **Split first** | `train_test_split` before any cleaning | so cleaning stats never see the test set (no leakage) |
| **Impute** | median (numeric), most-frequent (categorical) | keeps every row; dropping wastes good data and can bias it |
| **Ordinal encode** | `size` → ordered integers S<M<L<XL | the order carries real information |
| **One-hot encode** | `city` → one 0/1 column per category | avoids injecting a fake order into an unordered variable |
| **Scale** | StandardScaler (default) / MinMaxScaler (bounded) | stops large-magnitude features from dominating |
| **Pipeline + ColumnTransformer** | one object does it all | `fit` on train, `transform` on test — leakage-safe and reusable |

The single most important habit: **fit your preprocessing on training data only.** The `Pipeline` exists so you can't forget.